# EXP1 — Balanced training (WeightedRandomSampler)

Optimized notebook version (no EDA; saves figures to disk).

In [6]:

import os
import time
import csv
import numpy as np
import pandas as pd

# Windows OpenMP workaround (prevents crashes with some conda/torch/matplotlib stacks)
if os.name == "nt":
    os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

# Headless-safe plotting: save figures to disk
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)


class HistopathologicCancerDataset(Dataset):
    """
    Histopathologic Cancer Detection dataset class compatible with PyTorch.

    This dataset contains 96x96 pixel histopathologic images for binary 
    classification (0 = no cancer, 1 = metastatic cancer tissue present).

    Parameters
    ----------
    root_dir : str
        Root directory containing the dataset (should contain 'train' and/or 'test' folders).
    data_type : str, default='train'
        Type of dataset to load. Must be 'train' or 'test'.
    transform : callable, optional
        A function/transform to apply to each image (e.g., torchvision.transforms).
    labels_file : str, optional
        Path to the CSV file containing labels. Required if data_type='train'.
        Should have columns: 'id' (image filename without extension) and 'label' (0 or 1).

    Attributes
    ----------
    data_dir : str
        Path to the image directory (train or test).
    image_ids : list
        List of image IDs (filenames without extensions).
    labels : dict or None
        Dictionary mapping image IDs to labels (only for training data).
    transform : callable
        Transform to apply to images.

    Methods
    -------
    __len__()
        Returns the number of samples in the dataset.
    __getitem__(idx)
        Returns the image and label at the given index, applying the transform if specified.
    visualize(img, mean=(0.5,0.5,0.5), std=(0.5,0.5,0.5))
        Display a single image (tensor or NumPy array), un-normalizing if needed.
    get_class_distribution()
        Returns the distribution of classes in the dataset (only for training data).
    """

    def __init__(self, root_dir, data_type='train', transform=None, labels_file=None):
        """
        Initialize the dataset.

        Parameters
        ----------
        root_dir : str
            Root directory containing 'train' and/or 'test' folders
        data_type : str
            'train' or 'test'
        transform : callable, optional
            Transform to apply to images
        labels_file : str, optional
            Path to CSV file with labels (required for train)
        """
        self.root_dir = root_dir
        self.data_type = data_type
        self.transform = transform

        # Set the data directory based on type
        if data_type == 'train':
            self.data_dir = os.path.join(root_dir, 'train')

            # Load labels from CSV file
            if labels_file is None:
                # Try default location
                labels_file = os.path.join(root_dir, 'train_labels.csv')

            if not os.path.exists(labels_file):
                raise FileNotFoundError(
                    f"Labels file not found at {labels_file}. "
                    "Please provide the path to train_labels.csv"
                )

            # Read labels CSV
            labels_df = pd.read_csv(labels_file)

            # Create a dictionary mapping image_id to label
            self.labels = dict(zip(labels_df['id'], labels_df['label']))

            # Get list of image IDs from the labels file
            self.image_ids = list(self.labels.keys())

        elif data_type == 'test':
            self.data_dir = os.path.join(root_dir, 'test')
            self.labels = None

            # Get list of all image files in test directory
            self.image_ids = [
                f.replace('.tif', '') for f in os.listdir(self.data_dir) 
                if f.endswith('.tif')
            ]

        else:
            raise ValueError("data_type must be 'train' or 'test'")

        # Verify that the data directory exists
        if not os.path.exists(self.data_dir):
            raise FileNotFoundError(
                f"Data directory not found at {self.data_dir}. "
                f"Please ensure the dataset is properly extracted."
            )

    def __len__(self):
        """Return the total number of images in the dataset."""
        return len(self.image_ids)

    def __getitem__(self, idx):
        """
        Get a single sample from the dataset.

        Parameters
        ----------
        idx : int
            Index of the sample to retrieve

        Returns
        -------
        tuple
            (image, label) where image is a PIL Image or transformed tensor,
            and label is an integer (0 or 1) for train data, or the image_id for test data
        """
        # Get image ID
        img_id = self.image_ids[idx]

        # Construct image path - images are .tif format
        img_path = os.path.join(self.data_dir, f"{img_id}.tif")

        # Load image using PIL
        img = Image.open(img_path).convert('RGB')

        # Apply transform if specified
        if self.transform:
            img = self.transform(img)

        # Return image and label (for train) or image_id (for test)
        if self.data_type == 'train':
            label = self.labels[img_id]
            return img, label
        else:
            # For test data, return the image_id instead of a label
            return img, img_id

    def visualize(self, img, mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)):
        """
        Display a single image (tensor or NumPy array), denormalizing if needed.

        Parameters
        ----------
        img : torch.Tensor or np.ndarray or PIL.Image
            Image to visualize
        mean : tuple
            Mean values used for normalization
        std : tuple
            Standard deviation values used for normalization
        """
        if isinstance(img, torch.Tensor):
            # Convert tensor to NumPy
            npimg = img.cpu().numpy()

            # Check if tensor is CHW (3,H,W), then permute to HWC
            if npimg.shape[0] == 3 and len(npimg.shape) == 3:
                npimg = np.transpose(npimg, (1, 2, 0))  # CHW -> HWC

            # Denormalize
            mean_arr = np.array(mean).reshape(1, 1, 3)
            std_arr = np.array(std).reshape(1, 1, 3)
            npimg = npimg * std_arr + mean_arr
            npimg = np.clip(npimg, 0, 1)

            plt.imshow(npimg)
        elif isinstance(img, Image.Image):
            # PIL Image
            plt.imshow(img)
        else:
            # NumPy array, assume already HWC
            plt.imshow(img)

        plt.axis('off')
        plt.show()

    def get_class_distribution(self):
        """
        Get the distribution of classes in the dataset.
        Only available for training data.

        Returns
        -------
        dict
            Dictionary with class counts: {0: count_no_cancer, 1: count_cancer}
        """
        if self.data_type != 'train' or self.labels is None:
            raise ValueError("Class distribution only available for training data")

        label_values = list(self.labels.values())
        distribution = {
            0: label_values.count(0),
            1: label_values.count(1)
        }

        print(f"Class Distribution:")
        print(f"  No Cancer (0): {distribution[0]:,} images ({distribution[0]/len(label_values)*100:.2f}%)")
        print(f"  Cancer (1): {distribution[1]:,} images ({distribution[1]/len(label_values)*100:.2f}%)")

        return distribution




class CancerCNN(nn.Module):
    """
    CNN for binary classification of histopathologic images (96x96x3).

    Parameters
    ----------
    in_channels : int, default=3
        Number of input channels (RGB images).
    num_classes : int, default=2
        Number of output classes (binary: 0=no cancer, 1=cancer).
    conv_layers : list of tuples, default=[(32,3), (64,3), (128,3)]
        Each tuple defines (out_channels, kernel_size) for conv layers.
    pool_type : str, default='max'
        Type of pooling: 'max' or 'avg'.
    pool_kernel : int, default=2
        Kernel size for pooling.
    pool_stride : int, default=2
        Stride for pooling.
    fc_layers : list of int, default=[256, 128]
        Fully connected layer sizes before output.
    activation : str, default='relu'
        Activation function: 'relu', 'leaky_relu', or 'elu'.
    dropout : float, default=0.5
        Dropout probability for regularization.
    input_size : tuple, default=(96, 96)
        Input image dimensions (height, width).
    """

    def __init__(self,
                 in_channels=3,
                 num_classes=2,
                 conv_layers=[(32, 3), (64, 3), (128, 3)],
                 pool_type='max',
                 pool_kernel=2,
                 pool_stride=2,
                 fc_layers=[256, 128],
                 activation='relu',
                 dropout=0.5,
                 input_size=(96, 96)):

        super(CancerCNN, self).__init__()

        # Select activation function
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'leaky_relu':
            self.activation = nn.LeakyReLU()
        elif activation == 'elu':
            self.activation = nn.ELU()
        else:
            raise ValueError("Unsupported activation")

        # Build convolutional layers
        self.conv_blocks = nn.ModuleList()
        current_channels = in_channels

        for out_channels, kernel_size in conv_layers:
            conv_block = nn.Sequential(
                nn.Conv2d(current_channels, out_channels, kernel_size=kernel_size, padding=1),
                nn.BatchNorm2d(out_channels),  # Added batch normalization
                self.activation,
                nn.MaxPool2d(kernel_size=pool_kernel, stride=pool_stride) if pool_type == 'max' 
                    else nn.AvgPool2d(kernel_size=pool_kernel, stride=pool_stride)
            )
            self.conv_blocks.append(conv_block)
            current_channels = out_channels

        # Compute flattened size after conv + pool layers
        with torch.no_grad():
            dummy = torch.zeros(1, in_channels, input_size[0], input_size[1])
            x = dummy
            for block in self.conv_blocks:
                x = block(x)
            flattened_size = torch.flatten(x, start_dim=1).shape[1]

        # Build fully connected layers with dropout
        self.fc_layers = nn.ModuleList()
        input_size_fc = flattened_size

        for units in fc_layers:
            self.fc_layers.append(nn.Sequential(
                nn.Linear(input_size_fc, units),
                self.activation,
                nn.Dropout(dropout)  # Added dropout for regularization
            ))
            input_size_fc = units

        # Output layer
        self.output_layer = nn.Linear(input_size_fc, num_classes)

    def forward(self, x):
        """
        Forward pass through the network.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor of shape (batch_size, in_channels, height, width).

        Returns
        -------
        torch.Tensor
            Output logits of shape (batch_size, num_classes).
        """
        # Convolutional blocks
        for block in self.conv_blocks:
            x = block(x)

        # Flatten
        x = torch.flatten(x, 1)

        # Fully connected layers
        for fc in self.fc_layers:
            x = fc(x)

        # Output layer (no activation - we'll use CrossEntropyLoss)
        x = self.output_layer(x)

        return x




def plot_training_curves(train_losses, val_losses, train_accs, val_accs, save_path='training_curves.png'):
    """Plot training and validation curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    epochs = range(1, len(train_losses) + 1)

    # Loss plot
    ax1.plot(epochs, train_losses, 'b-', label='Training Loss')
    ax1.plot(epochs, val_losses, 'r-', label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot(epochs, train_accs, 'b-', label='Training Accuracy')
    ax2.plot(epochs, val_accs, 'r-', label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Saved training curves to {save_path}")




def metrics_from_confusion_matrix(cm, class_names=None):
    """
    Given a square confusion matrix cm (actual rows, predicted cols),
    compute per-class TP, FP, FN, TN and derived metrics:
      precision, recall (sensitivity), specificity, f1, support

    Parameters
    ----------
    cm : np.ndarray
        Confusion matrix
    class_names : list, optional
        Names of the classes

    Returns
    -------
    tuple
        (metrics_per_class, summary) where metrics_per_class is a dict
        and summary contains overall statistics
    """
    cm = np.array(cm, dtype=np.int64)
    n_classes = cm.shape[0]
    total = cm.sum()
    diag = np.diag(cm)
    metrics_per_class = {}
    eps = 1e-12

    for i in range(n_classes):
        TP = int(cm[i, i])
        FP = int(cm[:, i].sum() - TP)
        FN = int(cm[i, :].sum() - TP)
        TN = int(total - TP - FP - FN)

        precision = TP / (TP + FP + eps)
        recall = TP / (TP + FN + eps)  # sensitivity
        specificity = TN / (TN + FP + eps)
        f1 = 2 * precision * recall / (precision + recall + eps)
        support = int(cm[i, :].sum())

        name = class_names[i] if class_names is not None else str(i)
        metrics_per_class[name] = {
            'TP': TP, 'FP': FP, 'FN': FN, 'TN': TN,
            'precision': precision, 'recall': recall,
            'specificity': specificity, 'f1': f1, 'support': support
        }

    # Overall metrics
    accuracy = diag.sum() / (total + eps)
    # macro averages
    macro_precision = np.mean([m['precision'] for m in metrics_per_class.values()])
    macro_recall = np.mean([m['recall'] for m in metrics_per_class.values()])
    macro_f1 = np.mean([m['f1'] for m in metrics_per_class.values()])

    summary = {
        'accuracy': accuracy,
        'macro_precision': macro_precision,
        'macro_recall': macro_recall,
        'macro_f1': macro_f1,
        'total': int(total)
    }

    return metrics_per_class, summary




def save_metrics_csv(metrics_per_class, summary, csv_path):
    """
    Save per-class metrics and summary to a CSV file.

    Parameters
    ----------
    metrics_per_class : dict
        Dictionary with metrics for each class
    summary : dict
        Summary statistics
    csv_path : str
        Path to save the CSV file
    """
    fieldnames = ['class', 'TP', 'FP', 'FN', 'TN', 'precision', 'recall', 'specificity', 'f1', 'support']

    with open(csv_path, 'w', newline='') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames, delimiter=';')
        writer.writeheader()

        for cls, metrics in metrics_per_class.items():
            row = {'class': cls}
            for k in fieldnames[1:]:
                value = metrics[k]
                if isinstance(value, float):
                    value = round(value, 4)
                row[k] = value
            writer.writerow(row)

        # Empty row
        writer.writerow({})

        # Summary
        summary_row = {
            'class': 'SUMMARY',
            'precision': round(summary['macro_precision'], 4),
            'recall': round(summary['macro_recall'], 4),
            'f1': round(summary['macro_f1'], 4),
            'support': summary['total']
        }
        writer.writerow(summary_row)

    print(f"✓ Saved metrics CSV to {csv_path}")




def plot_confusion_matrix(cm, title, save_path, class_names=None):
    """
    Plot and save confusion matrix.

    Parameters
    ----------
    cm : np.ndarray
        Confusion matrix
    title : str
        Title for the plot
    save_path : str
        Path to save the figure
    class_names : list, optional
        Names of the classes
    """
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.colorbar()
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')

    thresh = cm.max() / 2. if cm.max() > 0 else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(int(cm[i, j]), 'd'),
                    horizontalalignment="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=12, fontweight='bold')

    if class_names is not None:
        plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha='right')
        plt.yticks(np.arange(len(class_names)), class_names)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved confusion matrix to {save_path}")




def plot_roc_curves(roc_curves, class_names=None, title="ROC Curves", save_path=None):
    """
    Plot multi-class ROC curves.

    Parameters
    ----------
    roc_curves : dict
        Dictionary with ROC data for each class
    class_names : list, optional
        Names of the classes
    title : str
        Title for the plot
    save_path : str, optional
        Path to save the figure
    """
    if roc_curves is None:
        print("No ROC curves to plot.")
        return

    plt.figure(figsize=(10, 8))

    for class_idx, data in roc_curves.items():
        fpr = data['fpr']
        tpr = data['tpr']
        auc_value = data['auc']
        name = class_names[class_idx] if class_names is not None else str(class_idx)
        plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_value:.2f})', linewidth=2)

    plt.plot([0, 1], [0, 1], 'k--', label='Random', linewidth=2)
    plt.xlabel('False Positive Rate', fontsize=12, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=12, fontweight='bold')
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(loc='lower right', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"✓ Saved ROC curves to {save_path}")
    else:
        plt.close()





def plot_precision_recall_curve(probs, labels, class_names, title, save_path):
    """
    Precision-Recall curve(s).
    - Binary: uses probs[:,1] as positive score.
    - Multi-class: one-vs-rest per class.
    """
    plt.figure()

    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if probs.ndim == 2 and probs.shape[1] == 2:
        y_score = probs[:, 1]
        precision, recall, _ = precision_recall_curve(labels, y_score)
        ap = average_precision_score(labels, y_score)
        plt.plot(recall, precision, label=f"AP={ap:.4f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(title)
        plt.legend(loc="best")
    else:
        # One-vs-rest
        n_classes = probs.shape[1]
        for i in range(n_classes):
            y_true = (labels == i).astype(int)
            y_score = probs[:, i]
            precision, recall, _ = precision_recall_curve(y_true, y_score)
            ap = average_precision_score(y_true, y_score)
            cname = class_names[i] if class_names is not None and i < len(class_names) else str(i)
            plt.plot(recall, precision, label=f"{cname} (AP={ap:.4f})")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(title)
        plt.legend(loc="best")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_score_distributions(probs, labels, title, save_path):
    """
    Histogram of predicted positive-class probabilities separated by true label.
    Only meaningful for binary classification.
    """
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if probs.ndim != 2 or probs.shape[1] != 2:
        return  # skip for non-binary

    y_score = probs[:, 1]
    scores_0 = y_score[labels == 0]
    scores_1 = y_score[labels == 1]

    plt.figure()
    plt.hist(scores_0, bins=40, alpha=0.7, label="True 0")
    plt.hist(scores_1, bins=40, alpha=0.7, label="True 1")
    plt.xlabel("Predicted P(class=1)")
    plt.ylabel("Count")
    plt.title(title)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_threshold_sweep(probs, labels, title, save_path):
    """
    Sweep thresholds in [0,1] for binary classification and plot precision/recall/F1 vs threshold.
    """
    probs = np.asarray(probs)
    labels = np.asarray(labels)

    if probs.ndim != 2 or probs.shape[1] != 2:
        return  # skip for non-binary

    y_score = probs[:, 1]
    thresholds = np.linspace(0.0, 1.0, 101)

    precisions, recalls, f1s = [], [], []
    eps = 1e-12

    for t in thresholds:
        y_pred = (y_score >= t).astype(int)
        tp = np.sum((y_pred == 1) & (labels == 1))
        fp = np.sum((y_pred == 1) & (labels == 0))
        fn = np.sum((y_pred == 0) & (labels == 1))

        p = tp / (tp + fp + eps)
        r = tp / (tp + fn + eps)
        f1 = 2 * p * r / (p + r + eps)

        precisions.append(p)
        recalls.append(r)
        f1s.append(f1)

    plt.figure()
    plt.plot(thresholds, precisions, label="Precision")
    plt.plot(thresholds, recalls, label="Recall")
    plt.plot(thresholds, f1s, label="F1")
    plt.xlabel("Threshold")
    plt.ylabel("Score")
    plt.title(title)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def plot_train_val_summary(train_metrics, val_metrics, title, save_path):
    """
    Compact comparison plot (Train vs Val): accuracy, macro_f1, macro_recall, roc_auc_overall.
    """
    keys = ["accuracy", "macro_f1", "macro_recall"]
    train_vals = [train_metrics["metrics_summary"].get(k, np.nan) for k in keys]
    val_vals = [val_metrics["metrics_summary"].get(k, np.nan) for k in keys]

    # Add ROC-AUC (might be None if ROC failed)
    keys.append("roc_auc")
    train_vals.append(train_metrics.get("roc_auc_overall", np.nan))
    val_vals.append(val_metrics.get("roc_auc_overall", np.nan))

    x = np.arange(len(keys))
    width = 0.35

    plt.figure()
    plt.bar(x - width/2, train_vals, width, label="Train")
    plt.bar(x + width/2, val_vals, width, label="Val")
    plt.xticks(x, keys)
    plt.ylim(0, 1.0)
    plt.ylabel("Score")
    plt.title(title)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()

class CancerTrainer:
    """
    Handles training for the cancer detection model.
    """

    def __init__(self, config):
        """
        Initialize trainer with configuration.

        Parameters
        ----------
        config : dict
            Configuration dictionary containing:
            - model: PyTorch model
            - data_path: path to dataset
            - batch_size: batch size for training
            - num_epochs: number of training epochs
            - learning_rate: learning rate for optimizer
            - device: 'cuda' or 'cpu'
            - num_workers: number of workers for data loading
        """
        self.model = config["model"]
        self.data_path = config["data_path"]
        self.batch_size = config.get("batch_size", 32)
        self.num_epochs = config.get("num_epochs", 30)
        self.lr = config.get("learning_rate", 0.001)
        self.device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
        self.num_workers = config.get("num_workers", 0)

        # Move model to device
        self.model = self.model.to(self.device)

        # Define loss and optimizer
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.lr)

        # Setup data
        self._setup_data(config)

    def _setup_data(self, config):
        """
        Setup datasets and dataloaders with train/val split.

        PREPROCESSING AND AUGMENTATION HAPPENS HERE!
        """
        # ========================================================================
        # DATA AUGMENTATION AND PREPROCESSING
        # ========================================================================
        # This is where we add data augmentation and preprocessing!

        # Training transform with augmentation
        train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),          # Random horizontal flip
            transforms.RandomVerticalFlip(p=0.5),            # Random vertical flip
            transforms.RandomRotation(degrees=20),            # Random rotation
            transforms.ColorJitter(                           # Random color adjustments
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            ),
            transforms.RandomAffine(                          # Random affine transformations
                degrees=0,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1)
            ),
            transforms.ToTensor(),                            # Convert to tensor
            transforms.Normalize(                             # Normalize (preprocessing)
                mean=[0.485, 0.456, 0.406],                  # ImageNet mean
                std=[0.229, 0.224, 0.225]                    # ImageNet std
            )
        ])

        # Validation/Test transform (no augmentation, only preprocessing)
        val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            )
        ])

        # Load full training dataset
        full_train_dataset = HistopathologicCancerDataset(
            root_dir=self.data_path,
            data_type='train',
            transform=train_transform,  # Augmentation applied here
            labels_file=os.path.join(self.data_path, 'train_labels.csv')
        )

                # ------------------------------------------------------------------
        # Hierarchical 3-way split (STRATIFIED): 64% Train / 16% Val / 20% Test
        # 1) Hold out 20% as HELD-OUT TEST (never seen during training/tuning)
        # 2) Split remaining 80% DEV into TRAIN (80% of dev) and VAL (20% of dev)
        # ------------------------------------------------------------------
        split_seed = int(config.get("split_seed", 42))
        test_frac = float(config.get("test_frac", 0.20))
        val_frac_of_dev = float(config.get("val_frac_of_dev", 0.20))

        rng = np.random.default_rng(split_seed)

        all_labels = np.array(
            [full_train_dataset.labels[_id] for _id in full_train_dataset.image_ids],
            dtype=int
        )
        idx_all = np.arange(len(full_train_dataset))

        # --- Stratified TEST split ---
        idx0 = idx_all[all_labels == 0]
        idx1 = idx_all[all_labels == 1]
        rng.shuffle(idx0)
        rng.shuffle(idx1)

        n_test0 = int(round(test_frac * len(idx0)))
        n_test1 = int(round(test_frac * len(idx1)))

        test_indices = np.concatenate([idx0[:n_test0], idx1[:n_test1]])
        dev_indices  = np.concatenate([idx0[n_test0:], idx1[n_test1:]])
        rng.shuffle(test_indices)
        rng.shuffle(dev_indices)

        # --- Stratified VAL split within DEV ---
        dev_labels = all_labels[dev_indices]
        dev0 = dev_indices[dev_labels == 0]
        dev1 = dev_indices[dev_labels == 1]
        rng.shuffle(dev0)
        rng.shuffle(dev1)

        n_val0 = int(round(val_frac_of_dev * len(dev0)))
        n_val1 = int(round(val_frac_of_dev * len(dev1)))

        val_indices = np.concatenate([dev0[:n_val0], dev1[:n_val1]])
        train_indices = np.concatenate([dev0[n_val0:], dev1[n_val1:]])
        rng.shuffle(train_indices)
        rng.shuffle(val_indices)

        # Create subsets (TRAIN uses augmentation via full_train_dataset)
        train_dataset = torch.utils.data.Subset(full_train_dataset, train_indices.tolist())

        # Validation / Test datasets use clean transforms (no augmentation)
        val_dataset = HistopathologicCancerDataset(
            root_dir=self.data_path,
            data_type='train',
            transform=val_transform,
            labels_file=os.path.join(self.data_path, 'train_labels.csv')
        )
        val_dataset = torch.utils.data.Subset(val_dataset, val_indices.tolist())

        test_dataset = HistopathologicCancerDataset(
            root_dir=self.data_path,
            data_type='train',
            transform=val_transform,
            labels_file=os.path.join(self.data_path, 'train_labels.csv')
        )
        test_dataset = torch.utils.data.Subset(test_dataset, test_indices.tolist())

        train_size, val_size, test_size = len(train_dataset), len(val_dataset), len(test_dataset)

        # --- EXP1: BALANCED TRAINING via WeightedRandomSampler (keeps baseline augmentation) ---
        # Balance is applied ONLY to the TRAIN loader. Validation stays untouched (natural distribution).

        # Indices of the train subset (relative to full_train_dataset)
        train_indices = train_dataset.indices

        # Build targets (0/1) without loading images
        train_targets = [
            full_train_dataset.labels[full_train_dataset.image_ids[i]]
            for i in train_indices
        ]
        train_targets_t = torch.tensor(train_targets, dtype=torch.long)

        # Inverse-frequency weights: rarer class gets higher weight
        class_counts = torch.bincount(train_targets_t, minlength=2)  # [n0, n1]
        class_weights = 1.0 / class_counts.float().clamp_min(1.0)
        sample_weights = class_weights[train_targets_t]

        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),  # keep epoch length comparable
            replacement=True                  # oversample minority when needed
        )

        print(f"[EXP1] Train class counts (original): {class_counts.tolist()} -> using sampler for ~balanced batches")

        
        from collections import Counter
        
        os.makedirs("results_exp1_balanced/plots", exist_ok=True)
        
        def _bar_with_labels(ax, counts_dict, title):
            labels = ["No Cancer (0)", "Cancer (1)"]
            values = np.array([counts_dict.get(0, 0), counts_dict.get(1, 0)], dtype=float)
            total = max(values.sum(), 1.0)
            perc = 100.0 * values / total
        
            bars = ax.bar(labels, values)
            ax.set_title(title)
            ax.set_ylabel("Samples")
            ax.set_ylim(0, max(values) * 1.25 + 1)
        
            # Annotazioni: count + %
            for b, v, p in zip(bars, values, perc):
                ax.text(
                    b.get_x() + b.get_width() / 2,
                    b.get_height(),
                    f"{int(v)}\n({p:.1f}%)",
                    ha="center",
                    va="bottom",
                )
        
            # explicit ratio
            ax.text(
                0.5, -0.22,
                f"Cancer ratio: {perc[1]:.1f}%  |  No-cancer ratio: {perc[0]:.1f}%",
                transform=ax.transAxes,
                ha="center",
                va="top",
                fontsize=10,
            )
        
        # Actual distribution of the training split (before applying the sampler)
        real_counts = Counter(train_targets)
        
        # Effective “per-epoch” distribution (after the sampler)
        sampled_positions = list(iter(sampler))  # una "epoca" di sampling
        sampled_targets = [train_targets[p] for p in sampled_positions]
        effective_counts = Counter(sampled_targets)
        
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        
        _bar_with_labels(
            axes[0],
            real_counts,
            "EXP1 — Real distribution (train split)\nDataset unchanged"
        )
        
        _bar_with_labels(
            axes[1],
            effective_counts,
            "EXP1 — Effective distribution (1 epoch via sampler)\nBalanced batches for training (oversampling)"
        )
        
        fig.suptitle("EXP1 signature: dataset is unchanged, training batches are balanced (Oversampling)", fontsize=14)
        plt.tight_layout()
        
        out_path = "results_exp1_balanced/plots/exp1_signature_real_vs_sampler.png"
        plt.savefig(out_path, dpi=200)
        plt.show()
        
        print("Saved:", out_path)


        # Create dataloaders
        self.train_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            sampler=sampler,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True if self.device == 'cuda' else False
        )

        # Clean (unbalanced) train loader for evaluation only (so training metrics aren't biased by oversampling)
        self.train_eval_loader = DataLoader(
            train_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True if self.device == 'cuda' else False
        )


        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True if self.device == 'cuda' else False
        )



        self.test_loader = DataLoader(
            test_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True if self.device == 'cuda' else False
        )
        print(f"Dataset split: {train_size} training, {val_size} validation, {test_size} held-out test")
        print(f"Batches per epoch: {len(self.train_loader)}")

    def train_epoch(self):
        """Train for one epoch."""
        self.model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in self.train_loader:
            images = images.to(self.device)
            labels = labels.to(self.device)

            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)

            # Backward pass
            loss.backward()
            self.optimizer.step()

            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(self.train_loader)
        epoch_acc = 100. * correct / total

        return epoch_loss, epoch_acc

    def validate(self):
        """Validate the model."""
        self.model.eval()
        val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in self.val_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        val_loss = val_loss / len(self.val_loader)
        val_acc = 100. * correct / total

        return val_loss, val_acc

    def train(self):
        """Train the model for all epochs."""
        print(f"\n{'='*70}")
        print(f"Starting Training")
        print(f"{'='*70}")
        print(f"Device: {self.device}")
        print(f"Epochs: {self.num_epochs}")
        print(f"Batch size: {self.batch_size}")
        print(f"Learning rate: {self.lr}")
        print(f"{'='*70}\n")

        train_losses = []
        train_accs = []
        val_losses = []
        val_accs = []

        best_val_acc = 0.0

        for epoch in range(self.num_epochs):
            epoch_start = time.time()

            # Train
            train_loss, train_acc = self.train_epoch()

            # Validate
            val_loss, val_acc = self.validate()

            epoch_time = time.time() - epoch_start

            # Store metrics
            train_losses.append(train_loss)
            train_accs.append(train_acc)
            val_losses.append(val_loss)
            val_accs.append(val_acc)

            # Print progress
            print(f"Epoch [{epoch+1}/{self.num_epochs}] "
                  f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | "
                  f"Time: {epoch_time:.2f}s")

            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(self.model.state_dict(), 'best_model.pth')
                print(f"  → New best validation accuracy: {val_acc:.2f}%")

        print(f"\n{'='*70}")
        print(f"Training Complete!")
        print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
        print(f"{'='*70}\n")

        # Store results
        self.train_losses = train_losses
        self.train_accs = train_accs
        self.val_losses = val_losses
        self.val_accs = val_accs

        return train_losses, val_losses, train_accs, val_accs




class CancerEvaluator:
    """
    Handles evaluation of the trained cancer detection model.
    """

    def __init__(self, trainer, class_names=['No Cancer', 'Cancer']):
        """
        Initialize evaluator.

        Parameters
        ----------
        trainer : CancerTrainer
            Trained model trainer object
        class_names : list
            Names of the classes
        """
        self.trainer = trainer
        self.model = trainer.model
        self.device = trainer.device
        self.class_names = class_names

        # Create results directory
        self.results_dir = "results_exp1_balanced"
        self.plots_dir = os.path.join(self.results_dir, "plots")
        os.makedirs(self.plots_dir, exist_ok=True)

    def evaluate_dataset(self, data_loader, dataset_name="Dataset"):
        """
        Evaluate model on a given dataset.

        Parameters
        ----------
        data_loader : DataLoader
            DataLoader for the dataset to evaluate
        dataset_name : str
            Name of the dataset (for printing)

        Returns
        -------
        tuple
            (all_probs, all_labels, all_preds)
        """
        print(f"\n{'='*70}")
        print(f"Evaluating on {dataset_name}")
        print(f"{'='*70}")

        self.model.eval()
        all_probs = []
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for images, labels in data_loader:
                images = images.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(images)
                probs = torch.softmax(outputs, dim=1)
                preds = outputs.argmax(dim=1)

                all_probs.append(probs.cpu())
                all_labels.append(labels.cpu())
                all_preds.append(preds.cpu())

        all_probs = torch.cat(all_probs, dim=0).numpy()
        all_labels = torch.cat(all_labels, dim=0).numpy()
        all_preds = torch.cat(all_preds, dim=0).numpy()

        return all_probs, all_labels, all_preds

    def compute_metrics(self, probs, labels, preds):
        """
        Compute all evaluation metrics.
        """
        # Basic accuracy
        accuracy = accuracy_score(labels, preds)

        # Classification report
        report = classification_report(labels, preds, target_names=self.class_names, digits=4)

        # Confusion matrix
        cm = confusion_matrix(labels, preds)

        # Per-class metrics
        metrics_per_class, metrics_summary = metrics_from_confusion_matrix(cm, self.class_names)

        # ROC curves - FIXED VERSION
        n_classes = len(self.class_names)
        roc_curves = {}

        try:
            if n_classes == 2:
                # Binary classification - special case
                # Use probability of positive class (cancer = class 1)
                fpr, tpr, _ = roc_curve(labels, probs[:, 1])
                auc_value = auc(fpr, tpr)

                # Store for both classes (same curve, inverted for class 0)
                roc_curves[0] = {
                    'fpr': 1 - tpr,  # Inverted TPR becomes FPR for class 0
                    'tpr': 1 - fpr,  # Inverted FPR becomes TPR for class 0
                    'auc': 1 - auc_value
                }
                roc_curves[1] = {
                    'fpr': fpr,
                    'tpr': tpr,
                    'auc': auc_value
                }

                # Overall ROC AUC
                roc_auc_overall = auc_value
            else:
                # Multi-class classification (if you ever expand)
                all_labels_bin = label_binarize(labels, classes=np.arange(n_classes))
                for i in range(n_classes):
                    fpr, tpr, _ = roc_curve(all_labels_bin[:, i], probs[:, i])
                    auc_value = auc(fpr, tpr)
                    roc_curves[i] = {'fpr': fpr, 'tpr': tpr, 'auc': auc_value}
                roc_auc_overall = roc_auc_score(all_labels_bin, probs, multi_class='ovr')

        except Exception as e:
            print(f"Warning: Could not compute ROC curves: {e}")
            roc_curves = None
            roc_auc_overall = float('nan')

        return {
            'accuracy': accuracy,
            'probs': probs,
            'labels': labels,
            'preds': preds,
            'report': report,
            'confusion_matrix': cm,
            'metrics_per_class': metrics_per_class,
            'metrics_summary': metrics_summary,
            'roc_curves': roc_curves,
            'roc_auc_overall': roc_auc_overall
        }

    def evaluate_all(self):
        """
        Evaluate model on Train / Validation / Held-out Test sets.

        Note: the held-out test split must never be used for training or hyperparameter tuning.
        """
        print("\n" + "="*70)
        print("COMPREHENSIVE MODEL EVALUATION")
        print("="*70)

        # Training (prefer a clean loader if available)
        train_loader = getattr(self.trainer, 'train_eval_loader', self.trainer.train_loader)
        train_probs, train_labels, train_preds = self.evaluate_dataset(train_loader, "Training Set")
        train_metrics = self.compute_metrics(train_probs, train_labels, train_preds)

        # Validation
        val_probs, val_labels, val_preds = self.evaluate_dataset(self.trainer.val_loader, "Validation Set")
        val_metrics = self.compute_metrics(val_probs, val_labels, val_preds)

        # Held-out Test (if provided by the trainer)
        test_metrics = None
        if hasattr(self.trainer, "test_loader") and self.trainer.test_loader is not None:
            test_probs, test_labels, test_preds = self.evaluate_dataset(self.trainer.test_loader, "Held-Out Test Set")
            test_metrics = self.compute_metrics(test_probs, test_labels, test_preds)
        else:
            print("\n[Warning] trainer.test_loader not found — skipping held-out test evaluation.")

        self.train_metrics = train_metrics
        self.val_metrics = val_metrics
        self.test_metrics = test_metrics

        # Print results
        self._print_results(train_metrics, val_metrics, test_metrics)

        # Save visualizations
        self._save_visualizations(train_metrics, val_metrics, test_metrics)

        # Save metrics to CSV
        self._save_metrics_to_csv(train_metrics, val_metrics, test_metrics)

        return train_metrics, val_metrics, test_metrics

    def _print_results(self, train_metrics, val_metrics, test_metrics=None):
        """Print evaluation results."""
        print("\n" + "="*70)
        print("TRAINING SET RESULTS")
        print("="*70)
        print(f"Accuracy: {train_metrics['accuracy']:.4f}")
        print(f"ROC AUC: {train_metrics['roc_auc_overall']:.4f}")
        print("\nClassification Report:")
        print(train_metrics['report'])

        print("\n" + "="*70)
        print("VALIDATION SET RESULTS")
        print("="*70)
        print(f"Accuracy: {val_metrics['accuracy']:.4f}")
        print(f"ROC AUC: {val_metrics['roc_auc_overall']:.4f}")
        print("\nClassification Report:")
        print(val_metrics['report'])

        if test_metrics is not None:
            print("\n" + "="*70)
            print("HELD-OUT TEST SET RESULTS")
            print("="*70)
            print(f"Accuracy: {test_metrics['accuracy']:.4f}")
            print(f"ROC AUC: {test_metrics['roc_auc_overall']:.4f}")
            print("\nClassification Report:")
            print(test_metrics['report'])

        print("\n" + "="*70)
        print("SUMMARY")
        print("="*70)
        print(f"Train Accuracy: {train_metrics['accuracy']:.4f}")
        print(f"Val Accuracy:   {val_metrics['accuracy']:.4f}")
        if test_metrics is not None:
            print(f"Test Accuracy:  {test_metrics['accuracy']:.4f}")

        gap_tv = train_metrics['accuracy'] - val_metrics['accuracy']
        if test_metrics is not None:
            gap_tt = train_metrics['accuracy'] - test_metrics['accuracy']
            print(f"Gap (Train-Val):  {gap_tv:.4f}")
            print(f"Gap (Train-Test): {gap_tt:.4f}")
            if gap_tt > 0.10:
                print("\nWARNING: Significant gap between train and test accuracy (possible overfitting).")
        else:
            print(f"Gap (Train-Val):  {gap_tv:.4f}")

        if gap_tv > 0.10:
            print("\nWARNING: Significant gap between train and validation accuracy (possible overfitting).")

    def _save_visualizations(self, train_metrics, val_metrics, test_metrics=None):
        """Save all visualization plots for Train/Val/(Test)."""
        print("\n" + "="*70)
        print("SAVING VISUALIZATIONS")
        print("="*70)

        splits = [
            ("train", "Training Set", train_metrics),
            ("val", "Validation Set", val_metrics),
        ]
        if test_metrics is not None:
            splits.append(("test", "Held-Out Test Set", test_metrics))

        for tag, title_name, metrics in splits:
            # Confusion matrix (raw)
            plot_confusion_matrix(
                metrics['confusion_matrix'],
                title=f'Confusion Matrix - {title_name}',
                save_path=os.path.join(self.plots_dir, f'confusion_matrix_{tag}.png'),
                class_names=self.class_names
            )

            # Confusion matrix (normalized)
            cm = metrics['confusion_matrix'].astype('float')
            denom = cm.sum(axis=1, keepdims=True)
            denom = np.where(denom == 0, 1.0, denom)
            cm_norm = cm / denom

            plot_confusion_matrix(
                cm_norm,
                title=f'Normalized Confusion Matrix - {title_name}',
                save_path=os.path.join(self.plots_dir, f'confusion_matrix_{tag}_norm.png'),
                class_names=self.class_names
            )

            # ROC curves
            plot_roc_curves(
                metrics.get('roc_curves', None),
                class_names=self.class_names,
                title=f'ROC Curves - {title_name}',
                save_path=os.path.join(self.plots_dir, f'roc_curves_{tag}.png')
            )

            # PR curve + score distribution + threshold sweep (binary only; functions no-op otherwise)
            if "probs" in metrics and "labels" in metrics:
                plot_precision_recall_curve(
                    metrics["probs"], metrics["labels"], self.class_names,
                    f"Precision-Recall - {title_name}",
                    os.path.join(self.plots_dir, f"pr_curve_{tag}.png")
                )
                plot_score_distributions(
                    metrics["probs"], metrics["labels"],
                    f"Score distribution - {title_name}",
                    os.path.join(self.plots_dir, f"score_dist_{tag}.png")
                )
                plot_threshold_sweep(
                    metrics["probs"], metrics["labels"],
                    f"Threshold sweep - {title_name}",
                    os.path.join(self.plots_dir, f"threshold_sweep_{tag}.png")
                )

    def _save_metrics_to_csv(self, train_metrics, val_metrics, test_metrics=None):
        """Save metrics to CSV files (Train/Val/(Test))."""
        print("\n" + "="*70)
        print("SAVING METRICS TO CSV")
        print("="*70)

        # Per-split CSVs
        save_metrics_csv(
            train_metrics['metrics_per_class'],
            train_metrics['metrics_summary'],
            os.path.join(self.results_dir, 'metrics_train.csv')
        )

        save_metrics_csv(
            val_metrics['metrics_per_class'],
            val_metrics['metrics_summary'],
            os.path.join(self.results_dir, 'metrics_val.csv')
        )

        if test_metrics is not None:
            save_metrics_csv(
                test_metrics['metrics_per_class'],
                test_metrics['metrics_summary'],
                os.path.join(self.results_dir, 'metrics_test.csv')
            )

        # Summary TXT
        summary_path = os.path.join(self.results_dir, "evaluation_summary.txt")
        with open(summary_path, "w") as f:
            f.write("CANCER DETECTION MODEL - EVALUATION SUMMARY\n")
            f.write("="*70 + "\n\n")

            def _write_block(name, m):
                f.write(f"{name}\n")
                f.write("-"*70 + "\n")
                f.write(f"Accuracy:        {m['accuracy']:.4f}\n")
                f.write(f"ROC AUC:         {m['roc_auc_overall']:.4f}\n")
                f.write(f"Macro Precision: {m['metrics_summary']['macro_precision']:.4f}\n")
                f.write(f"Macro Recall:    {m['metrics_summary']['macro_recall']:.4f}\n")
                f.write(f"Macro F1:        {m['metrics_summary']['macro_f1']:.4f}\n\n")

            _write_block("TRAINING SET", train_metrics)
            _write_block("VALIDATION SET", val_metrics)
            if test_metrics is not None:
                _write_block("HELD-OUT TEST SET", test_metrics)

            f.write("="*70 + "\n")
            f.write("Generated by CancerEvaluator\n")

        print(f"Saved evaluation summary to: {summary_path}")

def main():
    # Reproducibility (same as baseline)
    torch.manual_seed(42)
    np.random.seed(42)

    # Configuration (aligned with baseline notebook)
    config = {
        "data_path": DATA_PATH,  # <-- change if your dataset is elsewhere
        "batch_size": 64,
        "num_epochs": 30,
        "learning_rate": 0.001,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "num_workers": 4,
        "model": CancerCNN(
            in_channels=3,
            num_classes=2,
            conv_layers=[(32, 3), (64, 3), (128, 3)],
            fc_layers=[256, 128],
            dropout=0.5
        )
    }

    print("=" * 70)
    print("HISTOPATHOLOGIC CANCER DETECTION - CNN TRAINING (EXP1: BALANCED TRAIN)")
    print("=" * 70)

    trainer = CancerTrainer(config)
    trainer.train()

    # Save training curves (saved to disk)
    os.makedirs("results_exp1_balanced", exist_ok=True)
    plot_training_curves(
        trainer.train_losses,
        trainer.val_losses,
        trainer.train_accs,
        trainer.val_accs,
        save_path=os.path.join("results_exp1_balanced", "training_curves.png"),
    )

    evaluator = CancerEvaluator(trainer)
    evaluator.evaluate_all()



In [ ]:
# Run experiment
main()


HISTOPATHOLOGIC CANCER DETECTION - CNN TRAINING (EXP1: BALANCED TRAIN)
[EXP1] Train class counts (original): [83781, 57035] -> using sampler for ~balanced batches
Saved: results_exp1_balanced/plots/exp1_signature_real_vs_sampler.png
Dataset split: 140816 training, 35204 validation, 44005 held-out test
Batches per epoch: 2201

Starting Training
Device: cuda
Epochs: 30
Batch size: 64
Learning rate: 0.001

Epoch [1/30] Train Loss: 0.5177 | Train Acc: 76.26% | Val Loss: 0.4255 | Val Acc: 82.09% | Time: 35.76s
  → New best validation accuracy: 82.09%
Epoch [2/30] Train Loss: 0.4618 | Train Acc: 79.16% | Val Loss: 0.4047 | Val Acc: 82.81% | Time: 35.94s
  → New best validation accuracy: 82.81%
Epoch [3/30] Train Loss: 0.4386 | Train Acc: 80.39% | Val Loss: 0.5487 | Val Acc: 73.19% | Time: 36.20s
Epoch [4/30] Train Loss: 0.4254 | Train Acc: 81.34% | Val Loss: 0.4070 | Val Acc: 83.20% | Time: 36.17s
  → New best validation accuracy: 83.20%
Epoch [5/30] Train Loss: 0.3991 | Train Acc: 82.75% | 